## First small model

### Install needed libraries

In [ ]:
# %pip install torch torchvision torch_geometric pandas numpy scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


### Load data

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

path_flo = "C:/Users/flori/Desktop/pa-1_data/zurich_1.las"

las = laspy.read(path_flo)
df = pd.DataFrame({
    "X": las.x,
    "Y": las.y,
    "Z": las.z,
    "Intensity": las.intensity,
    "Classification": las.classification
})

# Show the first few rows
print(df.head())

# total number of points
print("\nTotal number of points:\n", len(df))

                                                   X  \
0  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
1  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
2  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
3  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
4  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   

                                                   Y  \
0  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
1  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
2  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
3  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
4  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   

                                                   Z  Intensity  \
0  [433.53000000000003, 426.62, 433.13, 417.98, 4...         71   
1  [433.53000000000003, 426.62, 433.13, 417.98, 4...         73   
2  [433.53000000000003, 426.62, 433.13, 417.98, 4...        114   
3  [433.53000000000003, 426.62, 433.13, 417.98, 4...      

### Basic PyTorch Model

In [3]:
# Compute ground height (approximate lowest Z value)
ground_height = np.min(las.z)

# Extract Features (XYZ, Intensity, Return Number, Height Above Ground)
X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  # Additional feature
    las.z - ground_height  # Height normalization (Z - Ground)
)).T.astype(np.float32)  # Convert to float32

# Extract classification labels and create a mapping
unique_labels = np.unique(las.classification)  # Get unique class values
label_map = {label: idx for idx, label in enumerate(unique_labels)}  # Map to 0-based index
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)  # Convert labels

# Take a larger random sample (e.g., 5 million points)
sample_indices = np.random.choice(X.shape[0], size=12000000, replace=False)
X = X[sample_indices]
y = y[sample_indices]

# === Balance Classes (Undersample Majority Classes) ===
min_class_count = min(np.bincount(y))  # Find smallest class size
balanced_X, balanced_y = [], []
for class_label in np.unique(y):
    class_indices = np.where(y == class_label)[0]
    sampled_indices = resample(class_indices, replace=True, n_samples=min_class_count, random_state=42)
    balanced_X.append(X[sampled_indices])
    balanced_y.append(y[sampled_indices])

X = np.vstack(balanced_X)
y = np.hstack(balanced_y)

# === Normalize Features ===
scaler = StandardScaler()
X = scaler.fit_transform(X)

# === Convert to PyTorch tensors ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# === Define Improved Neural Network ===
class PointCloudNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(PointCloudNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)  # Dropout to prevent overfitting

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)  # Dropout applied before final layer
        x = self.fc4(x)
        return x

# Initialize model
input_size = X_train.shape[1]  # Now using all features
num_classes = len(unique_labels)  # Number of unique classifications
model = PointCloudNet(input_size, num_classes)

# === Define Loss, Optimizer, and Learning Rate Scheduler ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  # AdamW instead of Adam
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Reduce LR every 10 epochs

# === Train the Model (50 Epochs) ===
num_epochs = 50
batch_size = 1024

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()  # Adjust learning rate
    
    if (epoch + 1) % 5 == 0:  # Print loss every 5 epochs
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# === Evaluate Model ===
with torch.no_grad():
    test_outputs = model(X_test)
    predicted = torch.argmax(test_outputs, axis=1)
    accuracy = (predicted == y_test).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch [5/50], Loss: 1.3359
Epoch [10/50], Loss: 1.1735
Epoch [15/50], Loss: 1.1018
Epoch [20/50], Loss: 1.0494
Epoch [25/50], Loss: 1.0139
Epoch [30/50], Loss: 0.9858
Epoch [35/50], Loss: 0.9671
Epoch [40/50], Loss: 0.9548
Epoch [45/50], Loss: 0.9450
Epoch [50/50], Loss: 0.9379
Test Accuracy: 0.6634


### Basic PointNet++ Model

In [5]:
# === Imports and Model Definition ===

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import laspy
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


class PointNetSetAbstraction(nn.Module):
    def __init__(self, in_channels, mlp_layers):
        super(PointNetSetAbstraction, self).__init__()
        layers = []
        last_channel = in_channels
        for out_channel in mlp_layers:
            layers.append(nn.Conv1d(last_channel, out_channel, kernel_size=1))
            layers.append(nn.BatchNorm1d(out_channel))
            layers.append(nn.ReLU())
            last_channel = out_channel
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

class PointNetPlusPlus(nn.Module):
    def __init__(self, num_classes):
        super(PointNetPlusPlus, self).__init__()
        self.sa1 = PointNetSetAbstraction(6, [64, 128, 256])
        self.sa2 = PointNetSetAbstraction(256, [256, 512, 1024])
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(p=0.3)
        
    def forward(self, x):
        x = self.sa1(x)
        x = self.sa2(x)
        x = torch.max(x, 2)[0]  # Global max pooling
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x




In [14]:
# === Load and Preprocess LAS File ===

las = laspy.read(path_flo)

ground_height = np.min(las.z)

X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  
    las.z - ground_height  
)).T.astype(np.float32)  

unique_labels = np.unique(las.classification)
label_map = {label: idx for idx, label in enumerate(unique_labels)}
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)

sample_indices = np.random.choice(X.shape[0], size=50000, replace=False)
X = X[sample_indices]
y = y[sample_indices]

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)  # Keep original shape
X_test = torch.tensor(X_test, dtype=torch.float32)  
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [15]:
# === Train PointNet++ Model ===

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PointNetPlusPlus(num_classes=len(unique_labels)).to(device)
X_train, X_test, y_train, y_test = X_train.to(device), X_test.to(device), y_train.to(device), y_test.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        X_batch, y_batch = batch
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Reshape input to (batch_size, num_features, num_points)
        X_batch = X_batch.unsqueeze(2)  # Add a dummy dimension for num_points

        optimizer.zero_grad()
        outputs = model(X_batch)  # Now correctly shaped
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")

    # Reduce Learning Rate after 20 epochs
    if epoch == 15:
        for param_group in optimizer.param_groups:
            param_group['lr'] = 0.0001


Epoch 1/30:   0%|          | 0/2500 [00:00<?, ?it/s]

Epoch 1/30: 100%|██████████| 2500/2500 [02:41<00:00, 15.43it/s]


Epoch [1/30], Loss: 1980.4978


Epoch 2/30: 100%|██████████| 2500/2500 [02:47<00:00, 14.96it/s]


Epoch [2/30], Loss: 1788.0371


...

Epoch [28/30], Loss: 1349.8530


Epoch 29/30: 100%|██████████| 2500/2500 [02:54<00:00, 14.34it/s]


Epoch [29/30], Loss: 1353.4645


Epoch 30/30: 100%|██████████| 2500/2500 [02:47<00:00, 14.89it/s]

Epoch [30/30], Loss: 1363.0135


In [17]:
# === Evaluate PointNet++ Model ===

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        X_batch, y_batch = batch
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Ensure correct shape (batch_size, num_features, num_points)
        X_batch = X_batch.unsqueeze(2)  # Add a dummy dimension for num_points
        
        outputs = model(X_batch)  # Pass correctly shaped input
        predicted = torch.argmax(outputs, dim=1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

print(f"Test Accuracy: {correct / total:.4f}")


Test Accuracy: 0.7559
